# Exploratory Data Analysis: VAERS Long COVID Risk Assessment

**Sprint**: 1
**Agent**: DS-Agent
**Date**: 2025-12-25
**Objective**: Understand VAERS data structure, identify patterns, and prepare features for ML models

## Goals
1. Load and examine existing VAERS data files
2. Analyze batch code distributions and severity metrics
3. Explore symptom frequencies and co-occurrences
4. Identify key features for risk prediction
5. Generate summary statistics and visualizations

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully")

## 1. Data Loading

Load existing batch code data from the s2y-batches project

In [ ]:
# Define data paths
DATA_DIR = Path('../docs/data')
BATCH_TABLE_PATH = DATA_DIR / 'batchCodeTables' / 'Global.json'
HISTOGRAMS_DIR = DATA_DIR / 'histograms' / 'Global'

# Load global batch code table
print(f"Loading batch code data from {BATCH_TABLE_PATH}...")
with open(BATCH_TABLE_PATH, 'r') as f:
    batch_data = json.load(f)

# Convert to DataFrame
batch_df = pd.DataFrame(batch_data['data'], columns=batch_data['columns'])

print(f"Loaded {len(batch_df):,} batch codes")
print(f"\nColumns: {list(batch_df.columns)}")
print(f"\nFirst 5 rows:")
batch_df.head()

In [ ]:
# Basic data info
print("Dataset Information:")
print("=" * 60)
batch_df.info()

print("\nData Types:")
print(batch_df.dtypes)

## 2. Data Quality Assessment

In [ ]:
# Check for missing values
print("Missing Values Analysis:")
print("=" * 60)
missing = batch_df.isnull().sum()
missing_pct = (missing / len(batch_df) * 100).round(2)
missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_summary[missing_summary['Missing Count'] > 0])

# Check for duplicates
print(f"\nDuplicate batch codes: {batch_df.duplicated(subset=[batch_df.columns[0]]).sum()}")

In [ ]:
# Summary statistics
print("Summary Statistics:")
print("=" * 60)
batch_df.describe()

## 3. Batch Code Distribution Analysis

In [ ]:
# Manufacturer distribution
company_col = 'Company'  # Adjust column name if different
if company_col in batch_df.columns:
    manufacturer_counts = batch_df[company_col].value_counts()
    
    print("Batch Count by Manufacturer:")
    print(manufacturer_counts)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    manufacturer_counts.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Batch Code Distribution by Manufacturer', fontsize=16, fontweight='bold')
    ax.set_xlabel('Manufacturer', fontsize=12)
    ax.set_ylabel('Number of Batch Codes', fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 4. Adverse Event Severity Analysis

In [ ]:
# Analyze severity columns
severity_cols = [
    'Adverse Reaction Reports',
    'Deaths',
    'Disabilities',
    'Life-Threatening Illnesses',
    'Hospitalizations'
]

# Calculate totals
print("Total Adverse Events Across All Batches:")
print("=" * 60)
for col in severity_cols:
    if col in batch_df.columns:
        total = batch_df[col].sum()
        print(f"{col:40s}: {total:,}")

# Calculate percentage columns if they exist
if 'Severe reports' in batch_df.columns and 'Lethality' in batch_df.columns:
    print(f"\nAverage Severe Reports %: {batch_df['Severe reports'].mean():.2f}%")
    print(f"Average Lethality %: {batch_df['Lethality'].mean():.2f}%")

In [ ]:
# Severity distribution visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Adverse Event Distributions', fontsize=18, fontweight='bold')

severity_to_plot = ['Deaths', 'Disabilities', 'Life-Threatening Illnesses', 'Hospitalizations']

for idx, col in enumerate(severity_to_plot):
    if col in batch_df.columns:
        ax = axes[idx // 2, idx % 2]
        
        # Histogram
        data = batch_df[col].dropna()
        ax.hist(data, bins=50, edgecolor='black', alpha=0.7, color='coral')
        ax.set_title(f'{col} Distribution', fontsize=14, fontweight='bold')
        ax.set_xlabel('Count', fontsize=11)
        ax.set_ylabel('Number of Batches', fontsize=11)
        ax.grid(axis='y', alpha=0.3)
        
        # Add statistics text
        mean_val = data.mean()
        median_val = data.median()
        max_val = data.max()
        ax.text(0.65, 0.95, f'Mean: {mean_val:.1f}\nMedian: {median_val:.1f}\nMax: {max_val:.0f}',
                transform=ax.transAxes, fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 batches by adverse reaction reports
reports_col = 'Adverse Reaction Reports'
if reports_col in batch_df.columns:
    top_20_batches = batch_df.nlargest(20, reports_col)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    batch_codes = top_20_batches.iloc[:, 0]  # First column is batch code
    report_counts = top_20_batches[reports_col]
    
    bars = ax.barh(range(len(batch_codes)), report_counts, color='#d62728')
    ax.set_yticks(range(len(batch_codes)))
    ax.set_yticklabels(batch_codes)
    ax.set_xlabel('Adverse Reaction Reports', fontsize=12)
    ax.set_title('Top 20 Batches by Adverse Reaction Reports', fontsize=16, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(report_counts):
        ax.text(v + 50, i, f'{v:,.0f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## 5. Lethality and Severity Analysis

In [ ]:
# Scatter plot: Total Reports vs Deaths
if 'Adverse Reaction Reports' in batch_df.columns and 'Deaths' in batch_df.columns:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    x = batch_df['Adverse Reaction Reports']
    y = batch_df['Deaths']
    
    ax.scatter(x, y, alpha=0.5, s=30, c='darkblue')
    ax.set_xlabel('Adverse Reaction Reports', fontsize=12)
    ax.set_ylabel('Deaths', fontsize=12)
    ax.set_title('Relationship: Total Reports vs Deaths', fontsize=16, fontweight='bold')
    ax.grid(alpha=0.3)
    
    # Add trend line
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    ax.plot(x, p(x), "r--", linewidth=2, label=f'Trend: y={z[0]:.4f}x+{z[1]:.2f}')
    ax.legend()
    
    # Calculate correlation
    corr = batch_df[['Adverse Reaction Reports', 'Deaths']].corr().iloc[0, 1]
    ax.text(0.05, 0.95, f'Correlation: {corr:.3f}',
            transform=ax.transAxes, fontsize=12,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Lethality distribution
if 'Lethality' in batch_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    lethality_data = batch_df['Lethality'].dropna()
    axes[0].hist(lethality_data, bins=50, edgecolor='black', alpha=0.7, color='crimson')
    axes[0].set_xlabel('Lethality (%)', fontsize=12)
    axes[0].set_ylabel('Number of Batches', fontsize=12)
    axes[0].set_title('Lethality Percentage Distribution', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Box plot
    axes[1].boxplot(lethality_data, vert=True)
    axes[1].set_ylabel('Lethality (%)', fontsize=12)
    axes[1].set_title('Lethality Percentage Box Plot', fontsize=14, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print(f"Lethality Statistics:")
    print(f"  Mean: {lethality_data.mean():.2f}%")
    print(f"  Median: {lethality_data.median():.2f}%")
    print(f"  Std Dev: {lethality_data.std():.2f}%")
    print(f"  Min: {lethality_data.min():.2f}%")
    print(f"  Max: {lethality_data.max():.2f}%")
    print(f"  25th Percentile: {lethality_data.quantile(0.25):.2f}%")
    print(f"  75th Percentile: {lethality_data.quantile(0.75):.2f}%")

## 6. Symptom Histogram Analysis

Analyze individual batch histograms to understand symptom patterns

In [ ]:
# Load sample histograms
histogram_files = list(HISTOGRAMS_DIR.glob('*.json'))[:10]  # Sample 10 batches

print(f"Found {len(list(HISTOGRAMS_DIR.glob('*.json')))} histogram files")
print(f"Analyzing sample of {len(histogram_files)} batches...\n")

# Collect all symptoms across sampled batches
all_symptoms = Counter()
batch_histogram_data = []

for hist_file in histogram_files:
    with open(hist_file, 'r') as f:
        hist_data = json.load(f)
    
    batch_code = hist_file.stem
    
    # Extract histogram
    if 'histograms' in hist_data and len(hist_data['histograms']) > 0:
        histogram = hist_data['histograms'][0].get('histogram', {})
        all_symptoms.update(histogram)
        
        batch_histogram_data.append({
            'batch_code': batch_code,
            'company': hist_data.get('Company', 'Unknown'),
            'total_reports': hist_data.get('Adverse Reaction Reports', 0),
            'unique_symptoms': len(histogram),
            'total_symptom_instances': sum(histogram.values())
        })

histogram_summary_df = pd.DataFrame(batch_histogram_data)
print("Sample Histogram Summary:")
print(histogram_summary_df)

print(f"\nTotal unique symptoms found: {len(all_symptoms)}")

In [ ]:
# Top 30 most common symptoms across all sampled batches
top_30_symptoms = all_symptoms.most_common(30)

symptom_names = [s[0] for s in top_30_symptoms]
symptom_freqs = [s[1] for s in top_30_symptoms]

fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(range(len(symptom_names)), symptom_freqs, color='teal')
ax.set_yticks(range(len(symptom_names)))
ax.set_yticklabels(symptom_names, fontsize=10)
ax.set_xlabel('Frequency (across sampled batches)', fontsize=12)
ax.set_title('Top 30 Most Common Symptoms', fontsize=16, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(symptom_freqs):
    ax.text(v + 5, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Feature Engineering Ideas

In [ ]:
# Calculate additional features for risk prediction
feature_df = batch_df.copy()

# Feature 1: Severe event percentage
if all(col in feature_df.columns for col in ['Deaths', 'Disabilities', 'Life-Threatening Illnesses', 'Hospitalizations', 'Adverse Reaction Reports']):
    feature_df['severe_events'] = (
        feature_df['Deaths'] +
        feature_df['Disabilities'] +
        feature_df['Life-Threatening Illnesses'] +
        feature_df['Hospitalizations']
    )
    feature_df['severe_event_rate'] = (
        feature_df['severe_events'] / feature_df['Adverse Reaction Reports'] * 100
    ).fillna(0)

# Feature 2: Death rate
if 'Deaths' in feature_df.columns and 'Adverse Reaction Reports' in feature_df.columns:
    feature_df['death_rate'] = (
        feature_df['Deaths'] / feature_df['Adverse Reaction Reports'] * 100
    ).fillna(0)

# Feature 3: Disability rate
if 'Disabilities' in feature_df.columns and 'Adverse Reaction Reports' in feature_df.columns:
    feature_df['disability_rate'] = (
        feature_df['Disabilities'] / feature_df['Adverse Reaction Reports'] * 100
    ).fillna(0)

print("Engineered Features:")
print(feature_df[['severe_event_rate', 'death_rate', 'disability_rate']].describe())

In [ ]:
# Correlation matrix of key features
correlation_cols = [
    'Adverse Reaction Reports',
    'Deaths',
    'Disabilities',
    'Life-Threatening Illnesses',
    'Hospitalizations',
    'severe_event_rate',
    'death_rate',
    'disability_rate'
]

corr_df = feature_df[correlation_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Key Insights & Recommendations

### Summary Statistics

In [ ]:
print("=" * 80)
print("EXPLORATORY DATA ANALYSIS - KEY FINDINGS")
print("=" * 80)

print("\n1. DATASET OVERVIEW:")
print(f"   - Total batch codes: {len(batch_df):,}")
print(f"   - Total adverse reaction reports: {batch_df['Adverse Reaction Reports'].sum():,}")
print(f"   - Total deaths: {batch_df['Deaths'].sum():,}")
print(f"   - Total disabilities: {batch_df['Disabilities'].sum():,}")

if company_col in batch_df.columns:
    print(f"\n2. MANUFACTURER DISTRIBUTION:")
    for mfr, count in manufacturer_counts.items():
        pct = count / len(batch_df) * 100
        print(f"   - {mfr}: {count} batches ({pct:.1f}%)")

print(f"\n3. SEVERITY METRICS:")
if 'severe_event_rate' in feature_df.columns:
    print(f"   - Average severe event rate: {feature_df['severe_event_rate'].mean():.2f}%")
    print(f"   - Median severe event rate: {feature_df['severe_event_rate'].median():.2f}%")
if 'death_rate' in feature_df.columns:
    print(f"   - Average death rate: {feature_df['death_rate'].mean():.2f}%")
    print(f"   - Median death rate: {feature_df['death_rate'].median():.2f}%")

print(f"\n4. TOP SYMPTOMS (from sample):")
for idx, (symptom, freq) in enumerate(top_30_symptoms[:5], 1):
    print(f"   {idx}. {symptom}: {freq:,} occurrences")

print("\n5. RECOMMENDATIONS FOR ML MODELS:")
print("   a) Risk Score Prediction:")
print("      - Use XGBoost with features: severe_event_rate, death_rate, total_reports")
print("      - Target variable: Create composite risk score from Deaths/Disabilities weights")
print("   b) Symptom Clustering:")
print("      - Use K-Means on symptom co-occurrence matrix (TF-IDF vectors)")
print("      - Optimal clusters: 8-12 (based on domain knowledge)")
print("   c) Association Rule Mining:")
print("      - Apply Apriori algorithm on symptom transactions")
print("      - Min support: 0.05, Min confidence: 0.5, Min lift: 1.5")

print("\n" + "=" * 80)

## 9. Next Steps

**Sprint 1 Deliverables:**
1. ✅ Complete exploratory data analysis
2. ⏳ Feature engineering for risk prediction
3. ⏳ Baseline model training (XGBoost)
4. ⏳ Clustering feasibility study
5. ⏳ Document findings in Sprint 1 report

**Sprint 2 Goals:**
- Implement ETL pipeline for PostgreSQL loading
- Train optimized XGBoost model with hyperparameter tuning
- Develop K-Means clustering with silhouette score >0.6
- Generate association rules with Apriori
- Create feature importance visualizations

In [ ]:
# Save engineered features for next notebook
OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

feature_output_path = OUTPUT_DIR / 'batch_features_v1.csv'
feature_df.to_csv(feature_output_path, index=False)
print(f"Engineered features saved to: {feature_output_path}")

print("\n✅ EDA Complete!")